# Phase 2b - Flood fine-tune with rehearsal (VisDrone + C2A mixed)

**Why a re-run:** the first fine-tune (v1, C2A only) made the detector worse:

| | Phase 1 | v1 (C2A only) |
|---|---|---|
| flood test, person mAP50 | 0.324 | 0.222 |
| VisDrone val, mAP50 | 0.699 | 0.372 |
| VisDrone val, person mAP50 | 0.588 | 0.186 |

C2A's people are pasted-in cut-outs (often oversized, lying, hard-edged); the Phase-1
model recognises only ~4 % of them. Training on them alone replaced the model's idea of
a "person" - **catastrophic forgetting** across a **synthetic-to-real gap**.

**Fix - rehearsal:** fine-tune on **VisDrone train (6,471 real) + C2A (2,079, one copy per
scene)** together, so real people are in every batch while flood backgrounds are learned.
Validation uses both val sets, so the chosen epoch must stay good on both.

Same comparison as v1: Phase 1 vs this model on the **real flood test photos** and
**VisDrone val**, same resolution - and v1's numbers alongside.

`Runtime > Change runtime type > GPU`. ~1.5 h on a T4. Run every cell once, top to bottom.
**Coming back to a finished run?** Cells 1-9, then **10c**.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = '/content/drive/MyDrive/FloodRescue'
RUNS, MODELS, OUT = f'{PROJECT}/runs', f'{PROJECT}/models', f'{PROJECT}/outputs/phase2'
os.makedirs(OUT, exist_ok=True)
BASE = f'{MODELS}/visdrone_base.pt'
V1 = f'{MODELS}/flood_finetuned.pt'           # the C2A-only run, for the side-by-side
C2A_ZIP = f'{PROJECT}/data/flood_c2a_coco.zip'
for p in [BASE, C2A_ZIP]:
    assert os.path.exists(p), f'missing {p}'
print('ok')

## 3. Install

In [ ]:
!pip -q install "ultralytics>=8.3.0"
import ultralytics, torch
ultralytics.checks()

## 4. Flood C2A (from the zip already on your Drive)

In [ ]:
import glob
SRC = '/content/datasets/flood_c2a_coco'
if not os.path.exists(f'{SRC}/train/_annotations.coco.json'):
    os.makedirs(SRC, exist_ok=True)
    !unzip -q "{C2A_ZIP}" -d {SRC}
for split in ['train', 'valid', 'test']:
    print(f"{split:6s} {len(glob.glob(f'{SRC}/{split}/*.jpg'))} images")
# expected: train 20790 | valid 541 | test 109

## 5. VisDrone train + val (~1.5 GB, local disk)

Same mirror as Phase 1.

In [ ]:
VD = '/content/datasets/VisDrone'
os.makedirs(VD, exist_ok=True)
VD_URL = 'https://github.com/ultralytics/assets/releases/download/v0.0.0'
for zipname in ['VisDrone2019-DET-train.zip', 'VisDrone2019-DET-val.zip']:
    if not os.path.isdir(f'{VD}/{zipname[:-4]}'):
        !curl -L -o /content/{zipname} {VD_URL}/{zipname}
        !unzip -q /content/{zipname} -d {VD}
        !rm /content/{zipname}
!echo "train:" && ls {VD}/VisDrone2019-DET-train/images | wc -l
!echo "val:"   && ls {VD}/VisDrone2019-DET-val/images   | wc -l

## 6. Write the scripts

In [ ]:
%%writefile /content/visdrone_to_yolo.py
"""
Phase 1 - Step 1: Convert VisDrone2019-DET annotations to YOLO format.

VisDrone annotation line:
    bbox_left,bbox_top,bbox_width,bbox_height,score,category,truncation,occlusion

VisDrone categories:
    0 ignored-regions  1 pedestrian  2 people   3 bicycle  4 car   5 van
    6 truck            7 tricycle    8 awning-tricycle     9 bus  10 motor
    11 others

Project class scheme (kept IDENTICAL for the Phase-2 flood fine-tune so the
YOLO detection head is never reinitialized):
    0 person   <- pedestrian, people
    1 vehicle  <- bicycle, car, van, truck, tricycle, awning-tricycle, bus, motor
    2 animal   <- (none in VisDrone; appears only in the flood dataset)

No images are copied or moved. Labels are written to <split>/labels/ next to
<split>/images/, which is exactly where Ultralytics looks for them.

Usage:
    python visdrone_to_yolo.py --root /content/datasets/VisDrone
"""

import argparse
import json
from collections import Counter
from pathlib import Path

from PIL import Image

VISDRONE_TO_OURS = {
    1: 0, 2: 0,                                  # person
    3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1,  # vehicle
}
CLASS_NAMES = ["person", "vehicle", "animal"]
SPLITS = {"train": "VisDrone2019-DET-train", "val": "VisDrone2019-DET-val"}


def convert_split(split_dir: Path, min_side_px: float = 2.0) -> dict:
    img_dir, ann_dir = split_dir / "images", split_dir / "annotations"
    lbl_dir = split_dir / "labels"
    if not img_dir.is_dir() or not ann_dir.is_dir():
        raise FileNotFoundError(f"expected images/ and annotations/ inside {split_dir}")
    lbl_dir.mkdir(exist_ok=True)

    stats = {
        "images": 0, "images_with_no_objects": 0, "boxes_kept": 0,
        "skipped_ignored_region": 0, "skipped_other_class": 0,
        "skipped_degenerate": 0, "per_class": Counter(),
        # COCO definition of a "small" object: area < 32*32 px
        "small_boxes_coco": 0, "medium_boxes_coco": 0, "large_boxes_coco": 0,
    }

    for img_path in sorted(img_dir.iterdir()):
        if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        ann_path = ann_dir / f"{img_path.stem}.txt"
        stats["images"] += 1
        with Image.open(img_path) as im:
            W, H = im.size

        lines = []
        if ann_path.exists():
            for raw in ann_path.read_text().splitlines():
                raw = raw.strip().rstrip(",")
                if not raw:
                    continue
                p = raw.split(",")
                if len(p) < 6:
                    continue
                x, y, w, h = (float(v) for v in p[:4])
                score, cat = int(float(p[4])), int(float(p[5]))

                if score == 0 or cat == 0:          # ignored region
                    stats["skipped_ignored_region"] += 1
                    continue
                if cat not in VISDRONE_TO_OURS:     # category 11 "others"
                    stats["skipped_other_class"] += 1
                    continue

                # clamp to image bounds, then drop slivers
                x1, y1 = max(0.0, x), max(0.0, y)
                x2, y2 = min(float(W), x + w), min(float(H), y + h)
                bw, bh = x2 - x1, y2 - y1
                if bw < min_side_px or bh < min_side_px:
                    stats["skipped_degenerate"] += 1
                    continue

                cls = VISDRONE_TO_OURS[cat]
                lines.append(
                    f"{cls} {(x1 + bw / 2) / W:.6f} {(y1 + bh / 2) / H:.6f} "
                    f"{bw / W:.6f} {bh / H:.6f}"
                )
                stats["boxes_kept"] += 1
                stats["per_class"][CLASS_NAMES[cls]] += 1
                area = bw * bh
                if area < 32 ** 2:
                    stats["small_boxes_coco"] += 1
                elif area < 96 ** 2:
                    stats["medium_boxes_coco"] += 1
                else:
                    stats["large_boxes_coco"] += 1

        (lbl_dir / f"{img_path.stem}.txt").write_text("\n".join(lines))
        if not lines:
            stats["images_with_no_objects"] += 1

    stats["per_class"] = dict(stats["per_class"])
    return stats


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True,
                    help="dir containing VisDrone2019-DET-train / -val")
    ap.add_argument("--yaml-out", default=None,
                    help="path for the generated data.yaml (default <root>/visdrone.yaml)")
    ap.add_argument("--stats-out", default=None,
                    help="path for the conversion stats JSON")
    args = ap.parse_args()

    root = Path(args.root).resolve()
    all_stats = {}
    for split, folder in SPLITS.items():
        print(f"[convert] {split}: {folder}")
        all_stats[split] = convert_split(root / folder)
        s = all_stats[split]
        print(f"  images={s['images']}  boxes={s['boxes_kept']}  "
              f"per_class={s['per_class']}  "
              f"small(COCO)={s['small_boxes_coco']} "
              f"({100 * s['small_boxes_coco'] / max(1, s['boxes_kept']):.1f}%)  "
              f"empty_images={s['images_with_no_objects']}")

    yaml_path = Path(args.yaml_out) if args.yaml_out else root / "visdrone.yaml"
    yaml_path.parent.mkdir(parents=True, exist_ok=True)
    yaml_path.write_text(
        "# Auto-generated by visdrone_to_yolo.py - Phase 1 base detector\n"
        f"path: {root.as_posix()}\n"
        f"train: {SPLITS['train']}/images\n"
        f"val: {SPLITS['val']}/images\n"
        "nc: 3\n"
        "names:\n"
        "  0: person\n"
        "  1: vehicle\n"
        "  2: animal   # unused in VisDrone; reserved for the Phase-2 flood fine-tune\n"
    )
    print(f"[convert] wrote {yaml_path}")

    stats_path = Path(args.stats_out) if args.stats_out else root / "visdrone_conversion_stats.json"
    stats_path.parent.mkdir(parents=True, exist_ok=True)
    stats_path.write_text(json.dumps(all_stats, indent=2))
    print(f"[convert] wrote {stats_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/train_visdrone.py
"""
Phase 1 - Step 2: Train the Model-1 BASE detector on VisDrone (YOLOv8).

Tuned for one-shot success on Colab:
  * imgsz / batch auto-selected from the GPU actually allocated
  * mosaic kept on (big win for small objects), disabled for the last epochs
  * save_period checkpointing + --resume, so a Colab disconnect costs nothing
  * writes metrics JSON + copies best.pt to models/ for Phase 2

Usage (Colab):
    python train_visdrone.py --data /content/datasets/VisDrone/visdrone.yaml \
                             --project /content/drive/MyDrive/FloodRescue/runs
    python train_visdrone.py ... --resume      # after a disconnect
"""

import argparse
import json
import shutil
from pathlib import Path

import torch
from ultralytics import YOLO

# GPU-name substring -> (imgsz, batch).  Fallback used for anything unlisted.
GPU_PRESETS = [
    ("a100", (1280, 16)),
    ("h100", (1280, 24)),
    ("l4",   (1024, 10)),
    ("v100", (1024, 10)),
    ("t4",   ( 960, 12)),
    ("4090", (1280, 12)),
]
FALLBACK = (1024, 8)


def pick_preset():
    if not torch.cuda.is_available():
        raise SystemExit("No CUDA GPU. In Colab: Runtime > Change runtime type > GPU.")
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    low = name.lower()
    for key, preset in GPU_PRESETS:
        if key in low:
            print(f"[gpu] {name} ({vram:.0f} GB) -> imgsz={preset[0]} batch={preset[1]}")
            return preset
    print(f"[gpu] {name} ({vram:.0f} GB) unlisted -> imgsz={FALLBACK[0]} batch={FALLBACK[1]}")
    return FALLBACK


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="path to visdrone.yaml")
    ap.add_argument("--model", default="yolov8s.pt")
    ap.add_argument("--project", default="runs/phase1",
                    help="put this on Google Drive so checkpoints survive a disconnect")
    ap.add_argument("--name", default="visdrone_base")
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--patience", type=int, default=15,
                    help="epochs without mAP improvement before early stop")
    ap.add_argument("--imgsz", type=int, default=0, help="0 = auto from GPU")
    ap.add_argument("--batch", type=int, default=0, help="0 = auto from GPU")
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--val-only", action="store_true",
                    help="skip training; just re-validate weights and rewrite metrics")
    ap.add_argument("--weights", default=None,
                    help="with --val-only: weights to validate (default <run>/weights/best.pt)")
    ap.add_argument("--export-to", default="models",
                    help="dir to copy best.pt into as visdrone_base.pt")
    args = ap.parse_args()

    auto_imgsz, auto_batch = pick_preset()
    imgsz = args.imgsz or auto_imgsz
    batch = args.batch or auto_batch

    run_dir = Path(args.project) / args.name
    last = run_dir / "weights" / "last.pt"

    best = None
    if args.val_only:
        best = Path(args.weights) if args.weights else run_dir / "weights" / "best.pt"
        if not best.exists():
            raise SystemExit(f"--val-only given but {best} not found")
        print(f"[val-only] skipping training, re-validating {best}")

        class _R:  # stand-in so the shared code below finds the run directory
            save_dir = run_dir
        results = _R()
    elif args.resume:
        if not last.exists():
            raise SystemExit(f"--resume given but {last} not found")
        print(f"[resume] {last}")
        model = YOLO(str(last))
        results = model.train(resume=True)
    else:
        model = YOLO(args.model)
        results = model.train(
            data=args.data,
            epochs=args.epochs,
            imgsz=imgsz,
            batch=batch,
            device=0,
            workers=args.workers,
            project=args.project,
            name=args.name,
            exist_ok=True,
            # --- optimisation ---
            optimizer="SGD",
            lr0=0.01,
            lrf=0.01,
            cos_lr=True,
            momentum=0.937,
            weight_decay=0.0005,
            warmup_epochs=3.0,
            patience=args.patience,
            # --- augmentation: biased toward small, dense aerial targets ---
            mosaic=1.0,
            close_mosaic=10,      # last 10 epochs clean, so val-time stats match
            scale=0.5,
            translate=0.1,
            fliplr=0.5,
            flipud=0.0,           # drone imagery is gravity-oriented
            degrees=0.0,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
            erasing=0.0,
            # --- runtime ---
            amp=True,
            cache=False,          # VisDrone at 1024+ will not fit in Colab RAM
            save_period=5,        # checkpoint every 5 epochs
            plots=True,
            seed=0,
            val=True,
        )

    save_dir = Path(results.save_dir if hasattr(results, "save_dir") else run_dir)
    if best is None:
        best = save_dir / "weights" / "best.pt"
        print(f"[train] done -> {best}")

    # ---- final validation at the trained resolution -------------------------
    print("[val] running final validation on the VisDrone val split")
    metrics = YOLO(str(best)).val(
        data=args.data, imgsz=imgsz, batch=batch, device=0,
        project=args.project, name=f"{args.name}_val", exist_ok=True, plots=True,
    )
    box = metrics.box
    # Ultralytics builds box.maps as `np.zeros(nc) + self.map`, so any class with
    # no instances in the val split is silently filled with the OVERALL mAP.
    # Reporting that verbatim would claim an 'animal' score the data cannot
    # support. Only classes in ap_class_index actually have measured values.
    present = {int(c) for c in box.ap_class_index}
    summary = {
        "weights": str(best),
        "imgsz": imgsz,
        "batch": batch,
        "mAP50-95": round(float(box.map), 4),
        "mAP50": round(float(box.map50), 4),
        "mAP75": round(float(box.map75), 4),
        "precision": round(float(box.mp), 4),
        "recall": round(float(box.mr), 4),
        "per_class_mAP50-95": {
            metrics.names[c]: (round(float(box.maps[c]), 4) if c in present else None)
            for c in range(len(box.maps))
        },
        "classes_with_no_val_instances": [
            metrics.names[c] for c in range(len(box.maps)) if c not in present
        ],
    }
    print(json.dumps(summary, indent=2))

    out = save_dir / "phase1_visdrone_metrics.json"
    out.write_text(json.dumps(summary, indent=2))
    print(f"[val] wrote {out}")

    if args.export_to:
        dest_dir = Path(args.export_to)
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest = dest_dir / "visdrone_base.pt"
        if not (dest.exists() and dest.samefile(best)):
            shutil.copy2(best, dest)
        shutil.copy2(out, dest_dir / "phase1_visdrone_metrics.json")
        print(f"[export] {dest_dir / 'visdrone_base.pt'}  <-- feed this to Phase 2")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/prepare_flood_c2a.py
"""
Phase 2 - Step 1: Convert the Roboflow "Flood C2A" COCO export to YOLO format.

Dataset: Flood C2A v2 "Just Original" (Roboflow: wsar/flood-c2a-8vj8c, CC BY 4.0),
the flood subset of C2A (Nihal et al., ICPR 2024 - reference [3]).

What the export contains, and what this script does about it:
  * train: 2,079 aerial flood backgrounds with pasted-in people, but Roboflow
    stored 10 augmented copies of each (20,790 files). Ultralytics augments on
    the fly anyway, so only --copies-per-source of them are kept (default 2).
  * valid: 541 images whose backgrounds ALL also appear in train (different
    pasted people). Fine for picking the best epoch, too optimistic to report.
  * test: 109 REAL flood/landslide photos, disjoint from train. This is the
    split the Phase-1 vs Phase-2 comparison is reported on.
  * every category ("human", "0", "1") is a person -> class 0. Roboflow's
    parent category carries no boxes; "1" is 34 stray boxes in valid.
  * no vehicle labels at all. Fine-tuning on that would teach the model that
    cars are background. With --vehicle-weights, the Phase-1 model's confident
    vehicle detections are added to train/val as pseudo-labels. Never to test:
    scoring a model against its own predictions is circular.

Class scheme is identical to Phase 1 (0 person, 1 vehicle, 2 animal), so the
detection head of visdrone_base.pt is reused, not reinitialized.

Usage (Colab):
    python prepare_flood_c2a.py --src /content/datasets/flood_c2a_coco \\
                                --out /content/datasets/flood_c2a \\
                                --vehicle-weights /content/drive/MyDrive/FloodRescue/models/visdrone_base.pt
"""

import argparse
import json
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

CLASS_NAMES = ["person", "vehicle", "animal"]
SPLITS = {"train": "train", "valid": "val", "test": "test"}   # export dir -> ours
PSEUDO_SPLITS = {"train", "val"}


def source_key(file_name: str) -> str:
    """Roboflow names augmented copies '<source>.rf.<hash>.jpg'."""
    return file_name.split(".rf.")[0]


def iou(a, b) -> float:
    """IoU of two (x1, y1, x2, y2) boxes."""
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def yolo_line(cls: int, x1, y1, x2, y2, W, H) -> str:
    return (f"{cls} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} "
            f"{(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}")


def size_bucket(stats, area):
    # COCO definition, same as Phase 1, so the two datasets are comparable
    if area < 32 ** 2:
        stats["small_boxes_coco"] += 1
    elif area < 96 ** 2:
        stats["medium_boxes_coco"] += 1
    else:
        stats["large_boxes_coco"] += 1


def convert_split(src: Path, dst: Path, copies_per_source: int, seed: int,
                  min_side_px: float = 2.0):
    """Returns (stats, {image path: [person xyxy boxes]}) for pseudo-labelling."""
    coco = json.loads((src / "_annotations.coco.json").read_text())
    cat_names = {c["id"]: c["name"] for c in coco["categories"]}

    images = sorted(coco["images"], key=lambda im: im["file_name"])
    by_source = defaultdict(list)
    for im in images:
        by_source[source_key(im["file_name"])].append(im)
    if copies_per_source > 0:
        rng = random.Random(seed)
        keep = []
        for key in sorted(by_source):
            group = by_source[key]
            keep += group if len(group) <= copies_per_source else rng.sample(group, copies_per_source)
        images = sorted(keep, key=lambda im: im["file_name"])

    anns = defaultdict(list)
    for a in coco["annotations"]:
        anns[a["image_id"]].append(a)

    for sub in ("images", "labels"):
        if (dst / sub).exists():
            shutil.rmtree(dst / sub)
        (dst / sub).mkdir(parents=True)

    stats = {
        "images_in_export": len(coco["images"]), "sources": len(by_source),
        "images": 0, "images_with_no_objects": 0, "boxes_kept": 0,
        "skipped_degenerate": 0, "source_categories": Counter(),
        "per_class": Counter(),
        "small_boxes_coco": 0, "medium_boxes_coco": 0, "large_boxes_coco": 0,
    }
    persons = {}
    for im in images:
        W, H = float(im["width"]), float(im["height"])
        lines, boxes = [], []
        for a in anns[im["id"]]:
            x, y, w, h = a["bbox"]
            x1, y1 = max(0.0, x), max(0.0, y)
            x2, y2 = min(W, x + w), min(H, y + h)
            if x2 - x1 < min_side_px or y2 - y1 < min_side_px:
                stats["skipped_degenerate"] += 1
                continue
            stats["source_categories"][cat_names[a["category_id"]]] += 1
            lines.append(yolo_line(0, x1, y1, x2, y2, W, H))
            boxes.append((x1, y1, x2, y2))
            stats["boxes_kept"] += 1
            stats["per_class"]["person"] += 1
            size_bucket(stats, (x2 - x1) * (y2 - y1))

        img_out = dst / "images" / im["file_name"]
        shutil.copy2(src / im["file_name"], img_out)
        (dst / "labels" / f"{img_out.stem}.txt").write_text("\n".join(lines))
        persons[img_out] = boxes
        stats["images"] += 1
        if not lines:
            stats["images_with_no_objects"] += 1

    stats["source_categories"] = dict(stats["source_categories"])
    stats["per_class"] = dict(stats["per_class"])
    return stats, persons


def add_vehicle_pseudo_labels(persons: dict, weights: str, conf: float, imgsz: int,
                              max_person_iou: float = 0.5, chunk: int = 32) -> dict:
    """Append the base model's confident vehicle boxes to existing label files."""
    import torch
    from ultralytics import YOLO

    model = YOLO(weights)
    device = 0 if torch.cuda.is_available() else "cpu"
    paths = sorted(persons)
    added = dropped_overlap = imgs_with_vehicle = 0
    for i in range(0, len(paths), chunk):
        batch = paths[i:i + chunk]
        results = model.predict([str(p) for p in batch], imgsz=imgsz, conf=conf,
                                classes=[1], device=device, verbose=False)
        for p, r in zip(batch, results):
            H, W = r.orig_shape
            new = []
            for x1, y1, x2, y2 in r.boxes.xyxy.tolist():
                # a "vehicle" sitting on a labelled person is the person, misread
                if any(iou((x1, y1, x2, y2), b) >= max_person_iou for b in persons[p]):
                    dropped_overlap += 1
                    continue
                new.append(yolo_line(1, x1, y1, x2, y2, W, H))
            if new:
                lbl = p.parent.parent / "labels" / f"{p.stem}.txt"
                old = lbl.read_text()
                lbl.write_text("\n".join(([old] if old else []) + new))
                added += len(new)
                imgs_with_vehicle += 1
        print(f"  pseudo-labelled {min(i + chunk, len(paths))}/{len(paths)}", end="\r")
    print()
    return {"vehicle_boxes_added": added, "images_with_vehicle": imgs_with_vehicle,
            "dropped_overlapping_person": dropped_overlap,
            "conf": conf, "imgsz": imgsz, "weights": weights}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True, help="Roboflow COCO export (has train/ valid/ test/)")
    ap.add_argument("--out", required=True, help="YOLO dataset output dir")
    ap.add_argument("--copies-per-source", type=int, default=2,
                    help="augmented copies kept per train source image (0 = all 10)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--vehicle-weights", default=None,
                    help="Phase-1 weights; adds vehicle pseudo-labels to train/val")
    ap.add_argument("--vehicle-conf", type=float, default=0.5)
    ap.add_argument("--imgsz", type=int, default=960, help="inference size for pseudo-labelling")
    ap.add_argument("--stats-out", default=None,
                    help="path for the conversion stats JSON (default <out>/flood_c2a_stats.json)")
    args = ap.parse_args()

    src, out = Path(args.src).resolve(), Path(args.out).resolve()
    all_stats = {"copies_per_source": args.copies_per_source, "seed": args.seed}
    for theirs, ours in SPLITS.items():
        print(f"[convert] {theirs} -> {ours}")
        # only train carries augmented copies; valid/test are kept whole
        cps = args.copies_per_source if ours == "train" else 0
        s, persons = convert_split(src / theirs, out / ours, cps, args.seed)
        print(f"  images={s['images']} (of {s['images_in_export']}, {s['sources']} sources)  "
              f"boxes={s['boxes_kept']}  from={s['source_categories']}  "
              f"small(COCO)={s['small_boxes_coco']} "
              f"({100 * s['small_boxes_coco'] / max(1, s['boxes_kept']):.1f}%)")
        if args.vehicle_weights and ours in PSEUDO_SPLITS:
            print(f"[pseudo] vehicles in {ours} from {args.vehicle_weights} (conf>={args.vehicle_conf})")
            s["pseudo_vehicle"] = add_vehicle_pseudo_labels(
                persons, args.vehicle_weights, args.vehicle_conf, args.imgsz)
            s["per_class"]["vehicle"] = s["pseudo_vehicle"]["vehicle_boxes_added"]
            print(f"  {s['pseudo_vehicle']}")
        all_stats[ours] = s

    yaml_path = out / "flood_c2a.yaml"
    yaml_path.write_text(
        "# Auto-generated by prepare_flood_c2a.py - Phase 2 flood fine-tune\n"
        f"path: {out.as_posix()}\n"
        "train: train/images\n"
        "val: val/images     # shares backgrounds with train: epoch selection only\n"
        "test: test/images   # real flood photos, person labels only: report on this\n"
        "nc: 3\n"
        "names:\n"
        "  0: person\n"
        "  1: vehicle   # pseudo-labels from the Phase-1 model (train/val only)\n"
        "  2: animal    # no instances in C2A; kept so the Phase-1 head is reused\n"
    )
    print(f"[convert] wrote {yaml_path}")

    stats_path = Path(args.stats_out) if args.stats_out else out / "flood_c2a_stats.json"
    stats_path.parent.mkdir(parents=True, exist_ok=True)
    stats_path.write_text(json.dumps(all_stats, indent=2))
    print(f"[convert] wrote {stats_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/train_flood.py
"""
Phase 2 - Step 2: Fine-tune the Phase-1 VisDrone model on Flood C2A, then
compare it against the Phase-1 model on the same test data.

Fine-tune, not retrain:
  * starts from models/visdrone_base.pt with the same nc=3 head
  * lr0 10x lower than Phase 1 (0.001 vs 0.01), short warmup, no bias-lr spike
  * same imgsz as Phase 1 for the GPU, so both models are scored at one resolution

Comparison written to phase2_flood_metrics.json (or <export-name>_metrics.json):
  * flood test (109 real photos, person only) - "does fine-tuning help?"
  * VisDrone val (optional, --visdrone-data) - "did it forget vehicles?"

Usage (Colab):
    python train_flood.py --data /content/datasets/flood_c2a/flood_c2a.yaml \\
                          --weights /content/drive/MyDrive/FloodRescue/models/visdrone_base.pt \\
                          --visdrone-data /content/datasets/VisDrone/visdrone_val.yaml \\
                          --project /content/drive/MyDrive/FloodRescue/runs
    python train_flood.py ... --resume      # after a disconnect
    python train_flood.py ... --val-only    # comparison only, no training
"""

import argparse
import json
import shutil
from pathlib import Path

from ultralytics import YOLO

from train_visdrone import pick_preset


def summarize(metrics) -> dict:
    box = metrics.box
    # As in Phase 1: box.maps fills classes with no instances with the overall
    # mAP. Only ap_class_index classes were actually measured. On the flood
    # test split that is person only.
    present = {int(c) for c in box.ap_class_index}
    return {
        "mAP50-95": round(float(box.map), 4),
        "mAP50": round(float(box.map50), 4),
        "mAP75": round(float(box.map75), 4),
        "precision": round(float(box.mp), 4),
        "recall": round(float(box.mr), 4),
        "per_class_mAP50-95": {
            metrics.names[c]: (round(float(box.maps[c]), 4) if c in present else None)
            for c in range(len(box.maps))
        },
        "per_class_mAP50": {
            metrics.names[c]: (round(float(box.all_ap[i, 0]), 4))
            for i, c in enumerate(box.ap_class_index)
        },
        "classes_with_no_instances": [
            metrics.names[c] for c in range(len(box.maps)) if c not in present
        ],
    }


def evaluate(weights, data, split, imgsz, batch, project, name):
    m = YOLO(str(weights)).val(
        data=data, split=split, imgsz=imgsz, batch=batch, device=0,
        project=project, name=name, exist_ok=True, plots=True,
    )
    return summarize(m)


def delta(after: dict, before: dict) -> dict:
    keys = ["mAP50-95", "mAP50", "mAP75", "precision", "recall"]
    d = {k: round(after[k] - before[k], 4) for k in keys}
    d["per_class_mAP50-95"] = {
        c: (round(v - before["per_class_mAP50-95"][c], 4)
            if v is not None and before["per_class_mAP50-95"].get(c) is not None else None)
        for c, v in after["per_class_mAP50-95"].items()
    }
    return d


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="path to flood_c2a.yaml")
    ap.add_argument("--weights", required=True, help="Phase-1 visdrone_base.pt")
    ap.add_argument("--visdrone-data", default=None,
                    help="VisDrone yaml whose val split is the forgetting check")
    ap.add_argument("--project", default="runs/phase2")
    ap.add_argument("--name", default="flood_finetune")
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--patience", type=int, default=8)
    ap.add_argument("--lr0", type=float, default=0.001)
    ap.add_argument("--freeze", type=int, default=0,
                    help="freeze the first N layers (10 = whole backbone); 0 = train all")
    ap.add_argument("--imgsz", type=int, default=0, help="0 = auto from GPU, as Phase 1")
    ap.add_argument("--batch", type=int, default=0, help="0 = auto from GPU")
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--val-only", action="store_true",
                    help="skip training; just rerun the comparison")
    ap.add_argument("--finetuned", default=None,
                    help="with --val-only: fine-tuned weights (default <run>/weights/best.pt)")
    ap.add_argument("--export-to", default="models",
                    help="dir to copy best.pt into as <export-name>.pt")
    ap.add_argument("--export-name", default="flood_finetuned",
                    help="file name for the exported weights and metrics (keeps runs apart)")
    ap.add_argument("--close-mosaic", type=int, default=5,
                    help="final epochs trained without mosaic")
    args = ap.parse_args()

    auto_imgsz, auto_batch = pick_preset()
    imgsz = args.imgsz or auto_imgsz
    batch = args.batch or auto_batch

    run_dir = Path(args.project) / args.name
    last = run_dir / "weights" / "last.pt"

    if args.val_only:
        best = Path(args.finetuned) if args.finetuned else run_dir / "weights" / "best.pt"
        if not best.exists():
            raise SystemExit(f"--val-only given but {best} not found")
        print(f"[val-only] skipping training, comparing {best}")
    else:
        if args.resume:
            if not last.exists():
                raise SystemExit(f"--resume given but {last} not found")
            print(f"[resume] {last}")
            results = YOLO(str(last)).train(resume=True)
        else:
            results = YOLO(args.weights).train(
                data=args.data,
                epochs=args.epochs,
                imgsz=imgsz,
                batch=batch,
                device=0,
                workers=args.workers,
                project=args.project,
                name=args.name,
                exist_ok=True,
                # --- optimisation: gentle, so VisDrone knowledge survives ---
                optimizer="SGD",
                lr0=args.lr0,
                lrf=0.1,
                cos_lr=True,
                momentum=0.937,
                weight_decay=0.0005,
                warmup_epochs=1.0,
                warmup_bias_lr=0.0,   # default 0.1 would be 100x lr0 here
                freeze=args.freeze or None,
                patience=args.patience,
                # --- augmentation: as Phase 1 ---
                mosaic=1.0,
                close_mosaic=args.close_mosaic,
                scale=0.5,
                translate=0.1,
                fliplr=0.5,
                flipud=0.0,           # Roboflow already baked flips/rotations in
                degrees=0.0,
                hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
                erasing=0.0,
                # --- runtime ---
                amp=True,
                cache=False,
                save_period=5,
                plots=True,
                seed=0,
                val=True,
            )
        best = Path(getattr(results, "save_dir", None) or run_dir) / "weights" / "best.pt"
        print(f"[train] done -> {best}")

    # ---- comparison: Phase-1 base vs Phase-2 fine-tuned, same data, same imgsz ----
    report = {
        "base_weights": args.weights, "finetuned_weights": str(best),
        "imgsz": imgsz, "batch": batch,
        "flood_test": {}, "visdrone_val": {},
    }
    print("[compare] flood test split (real photos, person only)")
    report["flood_test"]["base"] = evaluate(
        args.weights, args.data, "test", imgsz, batch, args.project, f"{args.name}_test_base")
    report["flood_test"]["finetuned"] = evaluate(
        best, args.data, "test", imgsz, batch, args.project, f"{args.name}_test_finetuned")
    report["flood_test"]["delta"] = delta(report["flood_test"]["finetuned"],
                                          report["flood_test"]["base"])

    if args.visdrone_data:
        print("[compare] VisDrone val (forgetting check)")
        report["visdrone_val"]["base"] = evaluate(
            args.weights, args.visdrone_data, "val", imgsz, batch, args.project,
            f"{args.name}_visdrone_base")
        report["visdrone_val"]["finetuned"] = evaluate(
            best, args.visdrone_data, "val", imgsz, batch, args.project,
            f"{args.name}_visdrone_finetuned")
        report["visdrone_val"]["delta"] = delta(report["visdrone_val"]["finetuned"],
                                                report["visdrone_val"]["base"])

    print(json.dumps(report, indent=2))
    metrics_name = ("phase2_flood_metrics.json" if args.export_name == "flood_finetuned"
                    else f"{args.export_name}_metrics.json")
    out = run_dir / metrics_name
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(report, indent=2))
    print(f"[compare] wrote {out}")

    t = report["flood_test"]
    print(f"\nflood test  person mAP@50: base {t['base']['mAP50']:.3f} -> "
          f"fine-tuned {t['finetuned']['mAP50']:.3f} ({t['delta']['mAP50']:+.3f})")
    if report["visdrone_val"]:
        v = report["visdrone_val"]
        print(f"VisDrone val       mAP@50: base {v['base']['mAP50']:.3f} -> "
              f"fine-tuned {v['finetuned']['mAP50']:.3f} ({v['delta']['mAP50']:+.3f})")

    if args.export_to:
        dest_dir = Path(args.export_to)
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest = dest_dir / f"{args.export_name}.pt"
        if not (dest.exists() and dest.samefile(best)):
            shutil.copy2(best, dest)
        shutil.copy2(out, dest_dir / metrics_name)
        print(f"[export] {dest}  <-- feed this to Phase 3")


if __name__ == "__main__":
    main()


## 7. Convert VisDrone (same converter as Phase 1)

In [ ]:
!python /content/visdrone_to_yolo.py --root {VD} --yaml-out {VD}/visdrone.yaml \
    --stats-out /content/visdrone_conversion_stats.json

## 8. Convert C2A - one copy per scene + vehicle pseudo-labels

`--copies-per-source 1` (v1 used 2): 2,079 C2A images next to 6,471 VisDrone images,
so roughly 3 real images for every synthetic one.

In [ ]:
DATA = '/content/datasets/flood_c2a_r'
!python /content/prepare_flood_c2a.py \
    --src {SRC} \
    --out {DATA} \
    --copies-per-source 1 \
    --vehicle-weights {BASE} \
    --vehicle-conf 0.5 \
    --stats-out {OUT}/flood_c2a_rehearsal_stats.json

## 9. The mixed dataset

In [ ]:
from pathlib import Path
MIX = '/content/datasets/flood_rehearsal.yaml'
Path(MIX).write_text(f'''# Phase 2b rehearsal: VisDrone + C2A mixed
path: /content/datasets
train:
  - VisDrone/VisDrone2019-DET-train/images
  - flood_c2a_r/train/images
val:
  - VisDrone/VisDrone2019-DET-val/images
  - flood_c2a_r/val/images
test: flood_c2a_r/test/images   # real flood photos - the reported comparison
nc: 3
names:
  0: person
  1: vehicle
  2: animal
''')
n = {k: len(glob.glob(p)) for k, p in {
    'VisDrone train': f'{VD}/VisDrone2019-DET-train/images/*.jpg',
    'C2A train':      f'{DATA}/train/images/*.jpg',
    'VisDrone val':   f'{VD}/VisDrone2019-DET-val/images/*.jpg',
    'C2A val':        f'{DATA}/val/images/*.jpg',
    'flood test':     f'{DATA}/test/images/*.jpg'}.items()}
print(n)
print(f"real : synthetic in training = {n['VisDrone train'] / n['C2A train']:.1f} : 1")
# expected: 6471 | 2079 | 548 | 541 | 109  ->  3.1 : 1

## 10. Fine-tune with rehearsal + compare

10 epochs (each ~6-7 min on a T4: 8,550 images), lr0 0.001, last 3 epochs without
mosaic. Writes `models/flood_rehearsal.pt` and `models/flood_rehearsal_metrics.json` -
v1's files are left untouched.

> **Run once.** Disconnected -> **10b**. Already finished -> **10c**.

In [ ]:
if os.path.exists(f'{MODELS}/flood_rehearsal.pt'):
    raise SystemExit('Already finished (models/flood_rehearsal.pt exists). Use 10b or 10c.')

!python /content/train_flood.py \
    --data {MIX} \
    --weights {BASE} \
    --visdrone-data {VD}/visdrone.yaml \
    --project {RUNS} \
    --name flood_rehearsal \
    --epochs 10 \
    --patience 4 \
    --close-mosaic 3 \
    --export-to {MODELS} \
    --export-name flood_rehearsal

### 10b. Resume after a disconnect (re-run cells 1-9 first)

In [ ]:
!python /content/train_flood.py \
    --data {MIX} --weights {BASE} --visdrone-data {VD}/visdrone.yaml \
    --project {RUNS} --name flood_rehearsal --resume \
    --export-to {MODELS} --export-name flood_rehearsal

### 10c. Finished - rerun the comparison only

In [ ]:
!python /content/train_flood.py \
    --data {MIX} --weights {BASE} --visdrone-data {VD}/visdrone.yaml \
    --project {RUNS} --name flood_rehearsal --val-only \
    --finetuned {MODELS}/flood_rehearsal.pt \
    --export-to {MODELS} --export-name flood_rehearsal

## 11. Results: Phase 1 vs v1 (C2A only) vs rehearsal

In [ ]:
import json
r = json.load(open(f'{MODELS}/flood_rehearsal_metrics.json'))
v1p = f'{MODELS}/phase2_flood_metrics.json'
v1 = json.load(open(v1p)) if os.path.exists(v1p) else None

def get(d, block, model, key, cls=None):
    x = d[block][model]
    return x['per_class_mAP50'].get(cls) if cls else x[key]

rows = [('flood test  person mAP50', 'flood_test', 'mAP50', None),
        ('flood test  mAP50-95',     'flood_test', 'mAP50-95', None),
        ('flood test  recall',       'flood_test', 'recall', None),
        ('VisDrone    mAP50',        'visdrone_val', 'mAP50', None),
        ('VisDrone    person mAP50', 'visdrone_val', None, 'person'),
        ('VisDrone    vehicle mAP50','visdrone_val', None, 'vehicle')]
print(f"{'':28s}{'Phase 1':>9s}{'v1 C2A':>9s}{'rehearsal':>11s}")
for name, block, key, cls in rows:
    b = get(r, block, 'base', key, cls)
    f = get(r, block, 'finetuned', key, cls)
    o = get(v1, block, 'finetuned', key, cls) if v1 else float('nan')
    print(f'{name:28s}{b:9.3f}{o:9.3f}{f:11.3f}')

from IPython.display import Image as IPImage, display
p = f'{RUNS}/flood_rehearsal/results.png'
if os.path.exists(p):
    display(IPImage(p, width=900))

## 12. Side-by-side on real test photos

In [ ]:
from ultralytics import YOLO
import random, cv2, matplotlib.pyplot as plt
models = [('Phase 1', BASE), ('v1 C2A only', V1), ('rehearsal', f'{MODELS}/flood_rehearsal.pt')]
models = [(n, p) for n, p in models if os.path.exists(p)]
picks = random.Random(1).sample(sorted(glob.glob(f'{DATA}/test/images/*.jpg')), 3)
fig, axes = plt.subplots(len(models), 3, figsize=(22, 7 * len(models)))
for row, (tag, w) in zip(axes, models):
    for ax, res in zip(row, YOLO(w).predict(picks, imgsz=r['imgsz'], conf=0.25, verbose=False)):
        ax.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
        ax.set_title(f'{tag}: {len(res.boxes)} detections'); ax.axis('off')
plt.tight_layout()
plt.savefig(f'{OUT}/phase1_v1_rehearsal_test.jpg', dpi=70)
plt.show()

## 13. Download

In [ ]:
from google.colab import files
files.download(f'{MODELS}/flood_rehearsal.pt')
files.download(f'{MODELS}/flood_rehearsal_metrics.json')
files.download(f'{OUT}/flood_c2a_rehearsal_stats.json')
files.download(f'{OUT}/phase1_v1_rehearsal_test.jpg')